# Fraud Detection Model — Training Notebook

**Note:** This notebook was originally trained on Google Colab using a 150,000 row 
sample of the IEEE-CIS Fraud Detection dataset. The processed data file is not included 
in this repository due to size constraints (693MB raw data). The trained model artifacts 
are saved in `/models` and are used directly by the FastAPI backend.
 
To retrain: download the IEEE-CIS dataset from Kaggle, sample 150,000 rows, 
save to `data/processed/fraud_sampled.csv` and run cells sequentially.

In [ ]:
import os
os.chdir('/home/atharvakatkar/personal_projects/fraud-detection')
print("Working directory:", os.getcwd())

In [ ]:
# IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

# PLOT SETTINGS
plt.rcParams['figure.figsize'] = (12,6)
sns.set_theme(style='whitegrid')

print("Libraries imported successfully")

In [ ]:
# LOAD SAMPLED DATA
df = pd.read_csv('data/processed/fraud_sampled.csv')

print(f"Shape: {df.shape}")
print(f"\nFraud distribution:")
print(df['isFraud'].value_counts())
print(f"\nFraud rate: {df['isFraud'].mean()*100:.2f}%")
print(f"\nColumns: {df.shape[1]}")
print(f"\nSample:")
df.head()

In [ ]:
# BASIC EDA
print("Missing values per column (top 20):")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)
print(missing_df[missing_df['missing_count'] > 0].head(20))

print(f"\nTotal columns with missing values: {(missing > 0).sum()}")
print(f"Total columns with >50% missing: {(missing_pct > 50).sum()}")

In [ ]:
# DROP HIGH MISSING COLUMNS AND BASIC CLEANING
# Drop columns with more than 50% missing
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns with >50% missing")
print(f"Remaining shape: {df.shape}")

# Check remaining missing values
remaining_missing = df.isnull().sum()
print(f"\nColumns still with missing values: {(remaining_missing > 0).sum()}")

# Fill remaining numeric nulls with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill remaining categorical nulls with mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"Missing values after imputation: {df.isnull().sum().sum()}")
print(f"Final shape: {df.shape}")

In [ ]:
# CLASS IMBALANCE VISUALISATION
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Count plot
axes[0].bar(['Legitimate', 'Fraud'], 
            df['isFraud'].value_counts().values,
            color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title('Transaction Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(df['isFraud'].value_counts().values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Percentage plot
axes[1].pie(
    df['isFraud'].value_counts().values,
    labels=['Legitimate (97.35%)', 'Fraud (2.65%)'],
    colors=['steelblue', 'coral'],
    autopct='%1.2f%%',
    startangle=90
)
axes[1].set_title('Fraud vs Legitimate Ratio')

plt.suptitle('Class Imbalance — IEEE-CIS Fraud Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# TRANSACTION AMOUNT ANALYSIS
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribution of transaction amounts by fraud
axes[0].hist(df[df['isFraud']==0]['TransactionAmt'], 
             bins=50, alpha=0.7, color='steelblue', 
             label='Legitimate', density=True)
axes[0].hist(df[df['isFraud']==1]['TransactionAmt'], 
             bins=50, alpha=0.7, color='coral', 
             label='Fraud', density=True)
axes[0].set_xlabel('Transaction Amount')
axes[0].set_ylabel('Density')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlim(0, 1000)
axes[0].legend()

# Boxplot
df_plot = df[df['TransactionAmt'] < 1000]
fraud_amounts = df_plot[df_plot['isFraud']==1]['TransactionAmt']
legit_amounts = df_plot[df_plot['isFraud']==0]['TransactionAmt']
axes[1].boxplot([legit_amounts, fraud_amounts], 
                labels=['Legitimate', 'Fraud'],
                patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_ylabel('Transaction Amount ($)')
axes[1].set_title('Transaction Amount — Fraud vs Legitimate')

plt.tight_layout()
plt.show()

print(f"Median legitimate transaction: ${df[df['isFraud']==0]['TransactionAmt'].median():.2f}")
print(f"Median fraud transaction: ${df[df['isFraud']==1]['TransactionAmt'].median():.2f}")

In [ ]:
# FEATURE ENGINEERING
df_model = df.copy()

# Encode categorical columns
cat_cols = df_model.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns to encode: ", len(cat_cols))
print(cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Separate features and target
X = df_model.drop(columns=['isFraud', 'TransactionID'])
y = df_model['isFraud']

print("Features shape: ", X.shape)
print("Target distribution: ", y.value_counts())

In [ ]:
# TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set: ", X_train.shape)
print("Testing set: ", X_test.shape)
print("Training set fraud rate: ", np.round(y_train.mean()*100,2))
print("Testing set fraud rate: ", np.round(y_test.mean()*100,2))

# APPLY SMOTE TO TRAINING SET ONLY
print("Applying SMOTE")
smote = SMOTE(random_state=42, sampling_strategy=0.2)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE training shape: ", X_train_smote.shape)
print("After SMOTE fraud rate: ", np.round(y_train_smote.mean()*100,2))
print("Fraud cases before SMOTE: ", y_train.sum())
print("Fraud cases before SMOTE: ", y_train_smote.sum())

In [ ]:
# TRAIN XGBOOST MODEL
print("Training XGBoost model")
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    scale_pos_weight=1,
    random_state=42,
    eval_metric='auc',
    verbosity=0
)

xgb_model.fit(
    X_train_smote, y_train_smote,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print("Training complete")

In [ ]:
# MODEL EVALUATION
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

print("=" * 20)
print("XGBOOST MODEL EVALUATION")
print("=" * 20)
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
xticklabels=['Legitimate', 'Fraud'],
yticklabels=['Legitimate', 'Fraud'])
plt.title('Confusion Matrix - XGBOOST')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# THRESHOLD TUNING
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Find threshold that gives recall of ~0.75
target_recall = 0.75
idx = np.argmin(np.abs(recall - target_recall))
optimal_threshold = thresholds[idx]

print(f"Default threshold (0.5):")
print(f"  Precision: {precision_recall_curve(y_test, y_pred_proba)[0][np.argmin(np.abs(thresholds-0.5))]:.3f}")
print(f"  Recall: {recall[np.argmin(np.abs(thresholds-0.5))]:.3f}")

print(f"Optimal threshold ({optimal_threshold:.3f}) for ~75% recall:")
print(f"  Precision: {precision[idx]:.3f}")
print(f"  Recall: {recall[idx]:.3f}")

# Apply optimal threshold
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)
print(f"Classification Report at optimal threshold:")
print(classification_report(y_test, y_pred_optimal, target_names=['Legitimate', 'Fraud']))

# Plot precision recall curve
plt.figure(figsize=(10, 6))
plt.plot(recall, precision, color='steelblue', linewidth=2)
plt.axvline(x=recall[idx], color='red', linestyle='--', 
            label=f'Optimal threshold = {optimal_threshold:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — XGBoost')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# FEATURE IMPORTANCE
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'], 
         feature_importance['importance'],
         color='steelblue', edgecolor='black')
plt.xlabel('Importance Score')
plt.title('Top 20 Feature Importances — XGBoost')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 most important features:")
print(feature_importance.head(10).to_string(index=False))

In [ ]:
# SAVE MODEL
import os
os.makedirs('models', exist_ok=True)

joblib.dump(xgb_model, 'models/fraud_model.pkl')

# Save feature names for the API
feature_names = X.columns.tolist()
joblib.dump(feature_names, 'models/feature_names.pkl')

# Save optimal threshold
joblib.dump(optimal_threshold, 'models/threshold.pkl')

print("Saved:")
print("- models/fraud_model.pkl")
print("- models/feature_names.pkl")
print("- models/threshold.pkl")